# Notebook 3: Ingeniería Básica — Pruebas Unitarias en Jupyter
### Módulo de Nivelación Python — Sesión 3 · material transversal (S00)

> **Audiencia:** Estudiantes que conocen la lógica de algoritmos pero nunca han escrito pruebas formales.  
> **Duración estimada:** 35–40 minutos de live coding.  
> **Objetivo:** Adoptar el patrón Preparar–Actuar–Afirmar, usar `unittest` directamente en Jupyter, y comprender por qué las pruebas son la primera línea de defensa al implementar Estructuras de Datos.

---
## Parte 1 — ¿Por qué Probar Estructuras de Datos?

### El problema de confiar solo en `print()`

Hasta ahora, probablemente verificas tu código así:

```python
resultado = buscar(lista, 42)
print(resultado)   # miro la pantalla y... parece correcto
```

Esto funciona para un caso. Pero una Estructura de Datos tiene **decenas de casos límite**:
- ¿Qué pasa si la lista está vacía?
- ¿Qué pasa si el elemento buscado no existe?
- ¿Qué pasa si hay duplicados?
- ¿Qué pasa con una lista de un solo elemento?
- ¿Qué pasa si el elemento está al principio, al final, en el medio?

Verificar esto a ojo en cada cambio es inviable. Una **suite de pruebas** lo hace automáticamente en milisegundos.

### Analogía con Java

En Java usarías JUnit:

```java
@Test
public void testBuscarElementoExistente() {
    int[] arr = {10, 20, 30};
    assertEquals(1, Buscador.buscar(arr, 20));  // índice esperado: 1
}
```

En Python usamos `unittest`, que es parte de la biblioteca estándar (no necesitas instalar nada). El equivalente exacto:

```python
import unittest

class TestBuscador(unittest.TestCase):
    def test_buscar_elemento_existente(self):
        arr = [10, 20, 30]
        self.assertEqual(1, buscar(arr, 20))  # índice esperado: 1
```

### El patrón Preparar–Actuar–Afirmar (AAA)

Cada prueba tiene exactamente tres fases:

```
┌─────────────────────────────────────────────────────────────┐
│  PREPARAR  (Arrange)  →  Crea el entorno: objetos, datos    │
│  ACTUAR    (Act)      →  Llama al código bajo prueba        │
│  AFIRMAR   (Assert)   →  Verifica que el resultado es       │
│                          exactamente lo que esperabas       │
└─────────────────────────────────────────────────────────────┘
```

Una prueba bien escrita tiene **un solo concepto por método**: prueba una cosa, falla por una razón.

---
## Parte 2 — `unittest` en Jupyter: El Truco Necesario

Jupyter Notebook tiene un detalle importante: `unittest.main()` normalmente llama a `sys.exit()` al terminar, lo que **mataría el kernel** del notebook.

La solución es pasarle dos argumentos especiales:

```python
unittest.main(
    argv=['first-arg-is-ignored'],  # evita que unittest lea sys.argv de Jupyter
    exit=False                      # evita que llame a sys.exit()
)
```

Este patrón es **el estándar** para usar `unittest` en notebooks. Lo usaremos en todas las celdas de prueba.

In [ ]:
# === Prueba mínima funcional: verificando que el entorno está listo ===
import unittest


class TestEntorno(unittest.TestCase):
    """Verifica que unittest funciona correctamente dentro de Jupyter."""

    def test_python_es_python(self) -> None:
        """Una prueba trivial para confirmar que el framework responde."""
        # PREPARAR
        valor_esperado = 4
        # ACTUAR
        valor_obtenido = 2 + 2
        # AFIRMAR
        self.assertEqual(valor_esperado, valor_obtenido)

    def test_listas_son_iguales_por_valor(self) -> None:
        """En Python, == en listas compara contenido, no referencias."""
        a = [1, 2, 3]
        b = [1, 2, 3]
        self.assertEqual(a, b)        # compara contenido
        self.assertIsNot(a, b)        # pero son objetos distintos en memoria


# El patrón obligatorio para ejecutar unittest dentro de Jupyter:
unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

---
## Parte 3 — El Sujeto de Prueba: Búsqueda Lineal

Vamos a implementar y probar exhaustivamente la función de **búsqueda lineal** (`busqueda_lineal`).  
Es un algoritmo sencillo — perfecto para aprender el proceso de testing — pero con suficientes casos límite para que las pruebas sean valiosas.

### Contrato de la función

```
busqueda_lineal(lista, objetivo)
  → Retorna el índice de la primera aparición de 'objetivo' en 'lista'
  → Retorna -1 si 'objetivo' no está en 'lista'
```

### Casos que debemos cubrir

| # | Caso | Entrada | Salida esperada |
|---|---|---|---|
| 1 | Elemento en el medio | `[10, 20, 30]`, buscar `20` | `1` |
| 2 | Elemento al inicio | `[10, 20, 30]`, buscar `10` | `0` |
| 3 | Elemento al final | `[10, 20, 30]`, buscar `30` | `2` |
| 4 | Elemento no existe | `[10, 20, 30]`, buscar `99` | `-1` |
| 5 | Lista vacía | `[]`, buscar `5` | `-1` |
| 6 | Lista con un elemento (existe) | `[7]`, buscar `7` | `0` |
| 7 | Lista con un elemento (no existe) | `[7]`, buscar `99` | `-1` |
| 8 | Duplicados → retorna el primero | `[5, 5, 5]`, buscar `5` | `0` |
| 9 | Tipos mixtos | `["a", 1, True]`, buscar `"a"` | `0` |

In [ ]:
# === Implementación de busqueda_lineal ===
# Primero implementamos la función. Luego escribimos las pruebas.
# (En TDD harías lo inverso: pruebas primero, luego implementación)

from typing import Any


def busqueda_lineal(lista: list[Any], objetivo: Any) -> int:
    """
    Busca 'objetivo' en 'lista' mediante recorrido secuencial.

    :param lista:    Lista de elementos a recorrer.
    :param objetivo: Valor a encontrar.
    :return:         Índice de la primera aparición de 'objetivo',
                     o -1 si no se encuentra.
    :complexity:     O(n) tiempo, O(1) espacio.
    """
    for indice, elemento in enumerate(lista):
        if elemento == objetivo:
            return indice
    return -1


# Smoke test rápido antes de las pruebas formales
print(busqueda_lineal([10, 20, 30], 20))  # esperado: 1
print(busqueda_lineal([10, 20, 30], 99))  # esperado: -1
print(busqueda_lineal([], 5))             # esperado: -1

---
## Parte 4 — La Suite de Pruebas Completa

### Métodos `assert` más usados en `unittest`

| Método | Verifica que... | Equivalente JUnit |
|---|---|---|
| `assertEqual(a, b)` | `a == b` | `assertEquals(a, b)` |
| `assertNotEqual(a, b)` | `a != b` | `assertNotEquals(a, b)` |
| `assertTrue(x)` | `x` es verdadero | `assertTrue(x)` |
| `assertFalse(x)` | `x` es falso | `assertFalse(x)` |
| `assertIsNone(x)` | `x is None` | `assertNull(x)` |
| `assertIsNotNone(x)` | `x is not None` | `assertNotNull(x)` |
| `assertIn(a, b)` | `a in b` | — |
| `assertRaises(Exc, fn, ...)` | `fn()` lanza `Exc` | `assertThrows(Exc, fn)` |
| `assertAlmostEqual(a, b)` | `a ≈ b` (floats) | — |

In [ ]:
# === Suite completa para busqueda_lineal ===
import unittest


class TestBusquedaLineal(unittest.TestCase):
    """
    Suite de pruebas para la función busqueda_lineal.
    Cubre: casos normales, bordes y tipos mixtos.
    """

    # ------------------------------------------------------------------
    # setUp: se ejecuta ANTES de cada prueba individual.
    # Equivale a @Before en JUnit 4 / @BeforeEach en JUnit 5.
    # Úsalo para preparar datos compartidos y evitar repetición.
    # ------------------------------------------------------------------
    def setUp(self) -> None:
        """Prepara las estructuras de datos reutilizadas en varias pruebas."""
        self.lista_tipica: list[int] = [10, 20, 30, 40, 50]
        self.lista_vacia: list[int] = []
        self.lista_un_elemento: list[int] = [7]
        self.lista_duplicados: list[int] = [5, 5, 5, 5]

    # ------------------------------------------------------------------
    # CASO 1: elemento en el medio
    # ------------------------------------------------------------------
    def test_elemento_en_el_medio(self) -> None:
        # PREPARAR (ya listo en setUp)
        # ACTUAR
        resultado = busqueda_lineal(self.lista_tipica, 30)
        # AFIRMAR
        self.assertEqual(2, resultado, msg="30 está en el índice 2")

    # ------------------------------------------------------------------
    # CASO 2: elemento al inicio (índice 0 — caso límite importante)
    # ------------------------------------------------------------------
    def test_elemento_al_inicio(self) -> None:
        resultado = busqueda_lineal(self.lista_tipica, 10)
        self.assertEqual(0, resultado, msg="10 está en el índice 0")

    # ------------------------------------------------------------------
    # CASO 3: elemento al final (recorre toda la lista)
    # ------------------------------------------------------------------
    def test_elemento_al_final(self) -> None:
        resultado = busqueda_lineal(self.lista_tipica, 50)
        self.assertEqual(4, resultado, msg="50 está en el índice 4")

    # ------------------------------------------------------------------
    # CASO 4: elemento no existe → retorna -1
    # ------------------------------------------------------------------
    def test_elemento_no_existe(self) -> None:
        resultado = busqueda_lineal(self.lista_tipica, 999)
        self.assertEqual(-1, resultado, msg="999 no está en la lista")

    # ------------------------------------------------------------------
    # CASO 5: lista vacía → nunca debe lanzar excepción
    # ------------------------------------------------------------------
    def test_lista_vacia(self) -> None:
        resultado = busqueda_lineal(self.lista_vacia, 5)
        self.assertEqual(-1, resultado, msg="Lista vacía siempre retorna -1")

    # ------------------------------------------------------------------
    # CASO 6 y 7: lista con un solo elemento
    # ------------------------------------------------------------------
    def test_un_elemento_que_existe(self) -> None:
        resultado = busqueda_lineal(self.lista_un_elemento, 7)
        self.assertEqual(0, resultado)

    def test_un_elemento_que_no_existe(self) -> None:
        resultado = busqueda_lineal(self.lista_un_elemento, 99)
        self.assertEqual(-1, resultado)

    # ------------------------------------------------------------------
    # CASO 8: duplicados → retorna el PRIMERO (índice 0)
    # ------------------------------------------------------------------
    def test_duplicados_retorna_primer_indice(self) -> None:
        resultado = busqueda_lineal(self.lista_duplicados, 5)
        self.assertEqual(0, resultado,
                         msg="Con duplicados debe retornar el índice menor")

    # ------------------------------------------------------------------
    # CASO 9: tipos mixtos — Python permite listas heterogéneas
    # ------------------------------------------------------------------
    def test_lista_tipos_mixtos(self) -> None:
        lista_mixta = ["hola", 42, 3.14, True, None]
        self.assertEqual(0, busqueda_lineal(lista_mixta, "hola"))
        self.assertEqual(1, busqueda_lineal(lista_mixta, 42))
        self.assertEqual(4, busqueda_lineal(lista_mixta, None))

    # ------------------------------------------------------------------
    # CASO 10: el retorno es un entero, nunca None ni bool
    # ------------------------------------------------------------------
    def test_tipo_de_retorno_es_entero(self) -> None:
        resultado = busqueda_lineal(self.lista_tipica, 10)
        self.assertIsInstance(resultado, int,
                              msg="El retorno siempre debe ser int")


# Ejecución del runner dentro de Jupyter
unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

---
## Parte 5 — Ver Fallar una Prueba (Red → Green → Refactor)

Una prueba que **siempre pasa** no te protege de nada. Necesitas ver que falla cuando el código está mal.  
El ciclo de TDD es:

```
  🔴 RED    → escribe la prueba, falla porque el código no existe
  🟢 GREEN  → escribe el mínimo código para que pase
  🔵 REFACTOR → mejora el código sin romper las pruebas
```

Vamos a simular una implementación **rota** y ver cómo las pruebas la detectan:

In [ ]:
# === Implementación ROTA (con bugs intencionales) ===
# Esta versión tiene dos bugs:
#   Bug 1: usa > en vez de >=, por lo que nunca encuentra el índice 0
#   Bug 2: retorna None en vez de -1 cuando no encuentra

def busqueda_lineal_rota(lista: list[Any], objetivo: Any) -> int:
    """Versión con bugs para demostrar fallos en pruebas."""
    for indice, elemento in enumerate(lista):
        if elemento == objetivo and indice > 0:  # Bug 1: excluye índice 0
            return indice
    # Bug 2: no retorna -1, retorna None implícitamente


class TestBusquedaLinealRota(unittest.TestCase):
    """Estas pruebas DEBEN fallar — así sabemos que detectan los bugs."""

    def test_detecta_bug_indice_cero(self) -> None:
        """Bug 1: la versión rota no encuentra elementos en índice 0."""
        # PREPARAR
        lista = [10, 20, 30]
        # ACTUAR
        resultado = busqueda_lineal_rota(lista, 10)
        # AFIRMAR  ← esta línea fallará, revelando el bug
        self.assertEqual(0, resultado,
                         msg="Bug detectado: índice 0 nunca se retorna")

    def test_detecta_bug_retorno_none(self) -> None:
        """Bug 2: la versión rota retorna None en vez de -1."""
        lista = [10, 20, 30]
        resultado = busqueda_lineal_rota(lista, 999)
        # assertEqual fallará porque None != -1
        self.assertEqual(-1, resultado,
                         msg="Bug detectado: se retorna None en vez de -1")


print("=== Ejecutando pruebas sobre la implementación ROTA ===")
print("Esperamos ver FALLOS — eso es lo correcto.\n")
unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

---
## Parte 6 — Segundo Sujeto: Inserción Ordenada

Ahora probamos un algoritmo con más lógica interna: **inserción ordenada** en una lista.  
La función recibe una lista ya ordenada y un nuevo valor, e inserta el valor en la posición correcta **sin usar `sort()`**.

Este patrón aparece directamente en estructuras como árboles BST e insertion sort.

### Contrato

```
insertar_ordenado(lista_ordenada, valor)
  → Modifica lista_ordenada insertando 'valor' en la posición correcta
  → La lista sigue ordenada de menor a mayor después de la inserción
  → Retorna None (modifica in-place, como append)
```

In [ ]:
# === Implementación de insertar_ordenado ===

def insertar_ordenado(lista: list[int], valor: int) -> None:
    """
    Inserta 'valor' en 'lista' manteniendo el orden ascendente.

    La inserción se realiza in-place usando list.insert().
    Recorre la lista hasta encontrar la posición donde
    el elemento anterior es menor y el siguiente es mayor o igual.

    :param lista: Lista de enteros ordenada ascendentemente.
                  Se modifica in-place.
    :param valor: Entero a insertar.
    :return:      None. La lista se modifica directamente.
    :complexity:  O(n) tiempo (búsqueda de posición + desplazamiento),
                  O(1) espacio adicional.
    """
    # Buscar la posición de inserción
    posicion: int = 0
    for i, elemento in enumerate(lista):
        if elemento >= valor:
            posicion = i
            break
        posicion = i + 1   # si todos son menores, insertar al final

    lista.insert(posicion, valor)


# Verificación rápida
mi_lista = [10, 20, 40, 50]
insertar_ordenado(mi_lista, 30)
print("Después de insertar 30:", mi_lista)  # [10, 20, 30, 40, 50]

insertar_ordenado(mi_lista, 5)
print("Después de insertar 5:", mi_lista)   # [5, 10, 20, 30, 40, 50]

insertar_ordenado(mi_lista, 100)
print("Después de insertar 100:", mi_lista) # [5, 10, 20, 30, 40, 50, 100]

In [ ]:
# === Suite completa para insertar_ordenado ===
import unittest


class TestInsertarOrdenado(unittest.TestCase):
    """
    Suite de pruebas para insertar_ordenado.
    Verifica invariante de orden en todos los casos.
    """

    def setUp(self) -> None:
        """Lista fresca para cada prueba (setUp crea una copia nueva)."""
        # IMPORTANTE: si usáramos un atributo mutable compartido entre pruebas,
        # el orden de ejecución importaría. setUp garantiza aislamiento.
        self.lista_base: list[int] = [10, 20, 40, 50]

    # Método auxiliar (helper): no es una prueba, pero reduce duplicación
    def _esta_ordenada(self, lista: list[int]) -> bool:
        """Retorna True si la lista está ordenada ascendentemente."""
        return all(lista[i] <= lista[i + 1] for i in range(len(lista) - 1))

    # ------------------------------------------------------------------
    # CASO 1: inserción en el medio
    # ------------------------------------------------------------------
    def test_insercion_en_el_medio(self) -> None:
        insertar_ordenado(self.lista_base, 30)
        # Verificar posición exacta
        self.assertEqual([10, 20, 30, 40, 50], self.lista_base)

    # ------------------------------------------------------------------
    # CASO 2: inserción al principio (valor menor que todos)
    # ------------------------------------------------------------------
    def test_insercion_al_inicio(self) -> None:
        insertar_ordenado(self.lista_base, 1)
        self.assertEqual(1, self.lista_base[0],
                         msg="El valor más pequeño debe quedar en índice 0")
        self.assertTrue(self._esta_ordenada(self.lista_base))

    # ------------------------------------------------------------------
    # CASO 3: inserción al final (valor mayor que todos)
    # ------------------------------------------------------------------
    def test_insercion_al_final(self) -> None:
        insertar_ordenado(self.lista_base, 999)
        self.assertEqual(999, self.lista_base[-1],
                         msg="El valor más grande debe quedar al final")
        self.assertTrue(self._esta_ordenada(self.lista_base))

    # ------------------------------------------------------------------
    # CASO 4: inserción en lista vacía
    # ------------------------------------------------------------------
    def test_insercion_en_lista_vacia(self) -> None:
        lista_vacia: list[int] = []
        insertar_ordenado(lista_vacia, 42)
        self.assertEqual([42], lista_vacia)

    # ------------------------------------------------------------------
    # CASO 5: valor duplicado — debe insertarse (no ignorarse)
    # ------------------------------------------------------------------
    def test_insercion_valor_duplicado(self) -> None:
        insertar_ordenado(self.lista_base, 20)  # 20 ya existe
        self.assertEqual(5, len(self.lista_base),
                         msg="Debe haber 5 elementos tras insertar duplicado")
        self.assertTrue(self._esta_ordenada(self.lista_base),
                         msg="La lista debe seguir ordenada con el duplicado")

    # ------------------------------------------------------------------
    # CASO 6: múltiples inserciones → invariante se mantiene siempre
    # ------------------------------------------------------------------
    def test_multiples_inserciones_mantienen_orden(self) -> None:
        valores_a_insertar = [35, 5, 100, 25, 45]
        for valor in valores_a_insertar:
            insertar_ordenado(self.lista_base, valor)
            # La lista debe estar ordenada después de CADA inserción
            self.assertTrue(
                self._esta_ordenada(self.lista_base),
                msg=f"Lista desordenada tras insertar {valor}: {self.lista_base}"
            )

    # ------------------------------------------------------------------
    # CASO 7: el tamaño crece exactamente en 1
    # ------------------------------------------------------------------
    def test_tamanio_incrementa_en_uno(self) -> None:
        tamanio_antes = len(self.lista_base)
        insertar_ordenado(self.lista_base, 30)
        self.assertEqual(tamanio_antes + 1, len(self.lista_base))

    # ------------------------------------------------------------------
    # CASO 8: la función retorna None (modifica in-place)
    # ------------------------------------------------------------------
    def test_retorna_none(self) -> None:
        resultado = insertar_ordenado(self.lista_base, 30)
        self.assertIsNone(resultado,
                          msg="insertar_ordenado no debe retornar ningún valor")


unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

---
## Parte 7 — Pruebas de Excepciones: `assertRaises`

Probar que el código **falla correctamente** es tan importante como probar que funciona bien.  
Una función robusta debe rechazar entradas inválidas con mensajes claros, no silenciosamente.

Aquí usamos el patrón de **context manager** con `with self.assertRaises(...)`, que es el equivalente a `@Test(expected = Exception.class)` en JUnit 4.

In [ ]:
# === Función con validación de entradas ===

def busqueda_binaria(lista: list[int], objetivo: int) -> int:
    """
    Busca 'objetivo' en una lista ordenada usando búsqueda binaria.

    :param lista:    Lista de enteros ordenada ascendentemente.
    :param objetivo: Entero a buscar.
    :return:         Índice del elemento, o -1 si no existe.
    :raises TypeError:  Si 'lista' no es una lista.
    :raises TypeError:  Si 'objetivo' no es un entero.
    :complexity:     O(log n) tiempo, O(1) espacio.
    """
    if not isinstance(lista, list):
        raise TypeError(f"Se esperaba list, se recibió {type(lista).__name__}")
    if not isinstance(objetivo, int):
        raise TypeError(f"Se esperaba int, se recibió {type(objetivo).__name__}")

    izq, der = 0, len(lista) - 1

    while izq <= der:
        medio = (izq + der) // 2
        if lista[medio] == objetivo:
            return medio
        elif lista[medio] < objetivo:
            izq = medio + 1
        else:
            der = medio - 1

    return -1


class TestBusquedaBinaria(unittest.TestCase):
    """Pruebas para busqueda_binaria, incluyendo casos de error."""

    def setUp(self) -> None:
        self.lista_par:   list[int] = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
        self.lista_impar: list[int] = [1, 3, 5, 7, 9]

    # Casos funcionales
    def test_encuentra_en_lista_par(self) -> None:
        self.assertEqual(5, busqueda_binaria(self.lista_par, 23))

    def test_encuentra_primer_elemento(self) -> None:
        self.assertEqual(0, busqueda_binaria(self.lista_par, 2))

    def test_encuentra_ultimo_elemento(self) -> None:
        self.assertEqual(9, busqueda_binaria(self.lista_par, 91))

    def test_no_encuentra_retorna_menos_uno(self) -> None:
        self.assertEqual(-1, busqueda_binaria(self.lista_par, 999))

    def test_lista_vacia(self) -> None:
        self.assertEqual(-1, busqueda_binaria([], 5))

    def test_lista_un_elemento_encontrado(self) -> None:
        self.assertEqual(0, busqueda_binaria([42], 42))

    def test_lista_un_elemento_no_encontrado(self) -> None:
        self.assertEqual(-1, busqueda_binaria([42], 99))

    # ------------------------------------------------------------------
    # Casos de ERROR: verificamos que las excepciones se lanzan
    # ------------------------------------------------------------------
    def test_lanza_error_si_lista_no_es_lista(self) -> None:
        """Pasar una tupla en vez de lista debe lanzar TypeError."""
        # Patrón context manager para assertRaises:
        with self.assertRaises(TypeError):
            busqueda_binaria((1, 2, 3), 2)   # tupla, no lista

    def test_lanza_error_si_objetivo_no_es_entero(self) -> None:
        """Pasar un float en vez de int debe lanzar TypeError."""
        with self.assertRaises(TypeError):
            busqueda_binaria([1, 2, 3], 2.5)  # float, no int

    def test_mensaje_error_menciona_tipo_recibido(self) -> None:
        """El mensaje de error debe indicar qué tipo se recibió."""
        # assertRaises como context manager captura la excepción en 'ctx'
        with self.assertRaises(TypeError) as ctx:
            busqueda_binaria("no soy lista", 5)
        # Verificamos que el mensaje es informativo
        self.assertIn("str", str(ctx.exception),
                      msg="El mensaje debe mencionar el tipo 'str' recibido")


unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

---
## Parte 8 — Pruebas sobre una Clase: `setUp` con Objetos

Cuando el sujeto de prueba es una clase (como `MatrizPersonalizada` del Notebook 2), `setUp` se convierte en el lugar donde se crean las instancias frescas para cada prueba.  
Esto garantiza que ningún test contamina el estado del siguiente.

In [ ]:
# === Clase mínima para demostrar pruebas sobre objetos ===
# (Versión simplificada de MatrizPersonalizada del Notebook 2)

from typing import Any


class Pila:
    """
    Estructura LIFO (Last In, First Out) respaldada por una lista Python.
    Esta es una versión adelantada del TDA completo del Notebook 4.
    """

    def __init__(self) -> None:
        self._elementos: list[Any] = []

    def apilar(self, elemento: Any) -> None:
        """Agrega 'elemento' al tope de la pila."""
        self._elementos.append(elemento)

    def desapilar(self) -> Any:
        """Elimina y retorna el elemento del tope."""
        if self.esta_vacia():
            raise IndexError("No se puede desapilar de una pila vacía")
        return self._elementos.pop()

    def tope(self) -> Any:
        """Retorna el elemento del tope sin eliminarlo."""
        if self.esta_vacia():
            raise IndexError("La pila está vacía")
        return self._elementos[-1]

    def esta_vacia(self) -> bool:
        """Retorna True si la pila no tiene elementos."""
        return len(self._elementos) == 0

    def tamanio(self) -> int:
        """Retorna el número de elementos en la pila."""
        return len(self._elementos)


class TestPila(unittest.TestCase):
    """
    Suite para la clase Pila.
    setUp crea una pila nueva antes de cada prueba → aislamiento total.
    """

    def setUp(self) -> None:
        """Instancia fresca de Pila para cada prueba."""
        self.pila = Pila()

    # ------------------------------------------------------------------
    # Estado inicial
    # ------------------------------------------------------------------
    def test_pila_nueva_esta_vacia(self) -> None:
        self.assertTrue(self.pila.esta_vacia())

    def test_pila_nueva_tiene_tamanio_cero(self) -> None:
        self.assertEqual(0, self.pila.tamanio())

    # ------------------------------------------------------------------
    # apilar
    # ------------------------------------------------------------------
    def test_apilar_incrementa_tamanio(self) -> None:
        self.pila.apilar(10)
        self.assertEqual(1, self.pila.tamanio())
        self.pila.apilar(20)
        self.assertEqual(2, self.pila.tamanio())

    def test_apilar_hace_pila_no_vacia(self) -> None:
        self.pila.apilar("x")
        self.assertFalse(self.pila.esta_vacia())

    # ------------------------------------------------------------------
    # tope
    # ------------------------------------------------------------------
    def test_tope_retorna_ultimo_apilado(self) -> None:
        self.pila.apilar(1)
        self.pila.apilar(2)
        self.pila.apilar(3)
        self.assertEqual(3, self.pila.tope())

    def test_tope_no_modifica_tamanio(self) -> None:
        self.pila.apilar(99)
        _ = self.pila.tope()
        self.assertEqual(1, self.pila.tamanio(),
                         msg="tope() no debe eliminar el elemento")

    # ------------------------------------------------------------------
    # desapilar: orden LIFO
    # ------------------------------------------------------------------
    def test_desapilar_retorna_en_orden_lifo(self) -> None:
        """LIFO: el último en entrar es el primero en salir."""
        for valor in [10, 20, 30]:
            self.pila.apilar(valor)
        self.assertEqual(30, self.pila.desapilar())
        self.assertEqual(20, self.pila.desapilar())
        self.assertEqual(10, self.pila.desapilar())

    def test_desapilar_decrementa_tamanio(self) -> None:
        self.pila.apilar(1)
        self.pila.apilar(2)
        self.pila.desapilar()
        self.assertEqual(1, self.pila.tamanio())

    # ------------------------------------------------------------------
    # Excepciones en pila vacía
    # ------------------------------------------------------------------
    def test_desapilar_pila_vacia_lanza_error(self) -> None:
        with self.assertRaises(IndexError):
            self.pila.desapilar()

    def test_tope_pila_vacia_lanza_error(self) -> None:
        with self.assertRaises(IndexError):
            self.pila.tope()


unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

---
## Resumen Visual

| Concepto | JUnit (Java) | unittest (Python) |
|---|---|---|
| Clase de prueba | `class Test extends TestCase` | `class Test(unittest.TestCase):` |
| Preparación por prueba | `@BeforeEach void setUp()` | `def setUp(self):` |
| Prueba individual | `@Test void testXxx()` | `def test_xxx(self):` |
| Igualdad | `assertEquals(a, b)` | `self.assertEqual(a, b)` |
| Verdadero/Falso | `assertTrue(x)` | `self.assertTrue(x)` |
| Nulo | `assertNull(x)` | `self.assertIsNone(x)` |
| Excepción | `assertThrows(E.class, fn)` | `with self.assertRaises(E):` |
| Ejecutar en Jupyter | N/A | `unittest.main(argv=[...], exit=False)` |

### Reglas de oro del testing

1. **Un concepto por prueba** — si el nombre tiene "y", probablemente son dos pruebas.
2. **Prueba los bordes siempre** — lista vacía, un elemento, duplicados.
3. **Una prueba que nunca falla no te protege** — verifica que falla cuando el código está roto.
4. **`setUp` garantiza aislamiento** — nunca compartas estado mutable entre pruebas.
5. **Prueba las excepciones** — el manejo de errores también tiene contrato.

---

## Ejercicios de Práctica

In [ ]:
# EJERCICIO 1
# Implementa la función contar_ocurrencias(lista, valor) que retorna
# cuántas veces aparece 'valor' en 'lista' (sin usar list.count()).
# Luego escribe una suite con al menos 5 casos de prueba.
# Casos sugeridos: lista vacía, sin ocurrencias, una ocurrencia,
#                  múltiples ocurrencias, todos iguales.

def contar_ocurrencias(lista: list[Any], valor: Any) -> int:
    """TU IMPLEMENTACIÓN AQUÍ."""
    pass


class TestContarOcurrencias(unittest.TestCase):
    def test_lista_vacia(self) -> None:
        pass  # TU PRUEBA AQUÍ

    def test_sin_ocurrencias(self) -> None:
        pass

    def test_una_ocurrencia(self) -> None:
        pass

    def test_multiples_ocurrencias(self) -> None:
        pass

    def test_todos_iguales(self) -> None:
        pass


unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

In [ ]:
# EJERCICIO 2 — Encuentra el bug con las pruebas
# La siguiente función tiene un bug. Sin mirar el código en detalle,
# escribe pruebas que lo detecten. Luego corrígelo.

def invertir_lista(lista: list[Any]) -> list[Any]:
    """Retorna una NUEVA lista con los elementos en orden inverso."""
    resultado = []
    for i in range(len(lista)):      # Bug: el rango es incorrecto
        resultado.append(lista[i])   # Bug: accede en orden normal
    return resultado


class TestInvertirLista(unittest.TestCase):
    def test_inversa_lista_tipica(self) -> None:
        pass  # TU PRUEBA — debe FALLAR con la implementación actual

    def test_lista_vacia(self) -> None:
        pass

    def test_lista_un_elemento(self) -> None:
        pass

    def test_no_modifica_original(self) -> None:
        """La función retorna nueva lista, no modifica la original."""
        pass


unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)

---
*Módulo de Nivelación Python — Notebook 3 de 4*  
*Siguiente: **Notebook 4 — TDAs y Contratos al Estilo Sedgewick***